# 06 Model Profile

Google Colab notebook version.

In [ ]:

# ============================================================
# COMPUTATIONAL FOOTPRINT AND INFERENCE-TIME PROFILING
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf

def run_model_profile(
    model,
    X_test,
    weights_path="model_weights.h5",
    model_path="model.keras",
):
    print("TensorFlow version:", tf.__version__)
    print("Model input shape:", model.input_shape)
    print("X_test shape:", X_test.shape)

    total_params = model.count_params()

    trainable_params = int(np.sum([
        np.prod(v.shape) for v in model.trainable_weights
    ]))

    non_trainable_params = int(np.sum([
        np.prod(v.shape) for v in model.non_trainable_weights
    ]))

    fp32_memory_mb = total_params * 4 / (1024 ** 2)
    fp16_memory_mb = total_params * 2 / (1024 ** 2)
    int8_memory_mb = total_params * 1 / (1024 ** 2)

    print("\n=== Model parameter footprint ===")
    print(f"Total parameters      : {total_params:,}")
    print(f"Trainable parameters  : {trainable_params:,}")
    print(f"Non-trainable params  : {non_trainable_params:,}")
    print(f"Approx. FP32 memory   : {fp32_memory_mb:.4f} MB")
    print(f"Approx. FP16 memory   : {fp16_memory_mb:.4f} MB")
    print(f"Approx. INT8 memory   : {int8_memory_mb:.4f} MB")

    model.save_weights(weights_path)

    full_model_size_mb = None
    try:
        model.save(model_path)
        full_model_size_mb = os.path.getsize(model_path) / (1024 ** 2)
    except Exception as e:
        print("Full model saving failed because of custom layer serialization.")
        print("Error:", e)

    weights_size_mb = os.path.getsize(weights_path) / (1024 ** 2)

    print("\n=== Disk footprint ===")
    print(f"Weights file size     : {weights_size_mb:.4f} MB")
    if full_model_size_mb is not None:
        print(f"Full Keras model size : {full_model_size_mb:.4f} MB")

    def benchmark_inference(model, X, batch_size=1, warmup_runs=30, timed_runs=200, device="/GPU:0"):
        x_sample = X[:batch_size].astype(np.float32)
        times = []

        with tf.device(device):
            for _ in range(warmup_runs):
                _ = model(x_sample, training=False).numpy()

            for _ in range(timed_runs):
                start = time.perf_counter()
                _ = model(x_sample, training=False).numpy()
                end = time.perf_counter()
                times.append(end - start)

        times = np.array(times)

        return {
            "Device": device,
            "Batch size": batch_size,
            "Mean latency per batch (ms)": times.mean() * 1000,
            "Std latency per batch (ms)": times.std() * 1000,
            "Median latency per batch (ms)": np.median(times) * 1000,
            "Mean latency per sample (ms)": (times.mean() * 1000) / batch_size,
            "Throughput (samples/s)": batch_size / times.mean(),
        }

    profiling_results = []

    available_gpus = tf.config.list_physical_devices("GPU")

    if len(available_gpus) > 0:
        for bs in [1, 16, 32, 64]:
            if bs <= len(X_test):
                profiling_results.append(
                    benchmark_inference(
                        model,
                        X_test,
                        batch_size=bs,
                        warmup_runs=30,
                        timed_runs=200,
                        device="/GPU:0",
                    )
                )

    for bs in [1, 16, 32]:
        if bs <= len(X_test):
            profiling_results.append(
                benchmark_inference(
                    model,
                    X_test,
                    batch_size=bs,
                    warmup_runs=20,
                    timed_runs=100,
                    device="/CPU:0",
                )
            )

    profiling_df = pd.DataFrame(profiling_results)

    print("\n=== Inference-time profiling ===")
    display(profiling_df)

    summary_df = pd.DataFrame([{
        "Total parameters": total_params,
        "Trainable parameters": trainable_params,
        "Non-trainable parameters": non_trainable_params,
        "Approx. FP32 parameter memory (MB)": fp32_memory_mb,
        "Approx. FP16 parameter memory (MB)": fp16_memory_mb,
        "Approx. INT8 parameter memory (MB)": int8_memory_mb,
        "Weights file size (MB)": weights_size_mb,
        "Full Keras model size (MB)": full_model_size_mb,
    }])

    print("\n=== Compact computational footprint summary ===")
    display(summary_df)

    summary_df.to_csv(summary_csv, index=False)
    profiling_df.to_csv(profiling_csv, index=False)

    print("\nSaved:")
    print(summary_csv)
    print(profiling_csv)

    print("\n=== LaTeX-ready footprint rows ===")
    print(f"Total parameters & {total_params:,} \\\\")
    print(f"Trainable parameters & {trainable_params:,} \\\\")
    print(f"Non-trainable parameters & {non_trainable_params:,} \\\\")
    print(f"Approx. FP32 parameter memory & {fp32_memory_mb:.4f} MB \\\\")
    print(f"Weights file size & {weights_size_mb:.4f} MB \\\\")

    print("\n=== LaTeX-ready inference rows ===")
    for _, row in profiling_df.iterrows():
        print(
            f"{row['Device']} & "
            f"{int(row['Batch size'])} & "
            f"{row['Mean latency per batch (ms)']:.3f} & "
            f"{row['Mean latency per sample (ms)']:.3f} & "
            f"{row['Throughput (samples/s)']:.2f} \\\\"
        )

    return summary_df, profiling_df
